### SQL-Based Exploratory Data Analysis (EDA)

This notebook uses **SQL (DuckDB) in Jupyter Notebook** to explore the PaySim financial transaction dataset and identify patterns associated with fraudulent transactions. It analyses fraud rates across transaction types, evaluates existing fraud detection rules, and investigates account balance anomalies, transaction amounts, and potential money-mule activities. The findings provide insights for **feature engineering and subsequent machine learning model development**.

**Note:** This notebook focuses on exploratory data analysis and does not train machine learning models.

### Load the dataset

In [6]:
import pandas as pd
import duckdb

# Read CSV
df = pd.read_csv("paysim_dataset.csv")

# Connect to DuckDB
con = duckdb.connect()

# Register DataFrame as SQL table
con.register("transactions", df)

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(6362620, 11)


### Verify data load
Expected: **6,362,620 rows** and **8,213 frauds**. This checks the dataset, not model performance.

In [7]:
display(con.execute("SELECT COUNT(*) AS total_transactions, SUM(isFraud) AS total_frauds FROM transactions").df())
display(con.execute("SELECT type, COUNT(*) AS total_transactions FROM transactions GROUP BY type ORDER BY type").df())

,total_transactions,total_frauds
0,6362620,8213.0


,type,total_transactions
0,CASH_IN,1399284
1,CASH_OUT,2237500
2,DEBIT,41432
3,PAYMENT,2151495
4,TRANSFER,532909


## 1.1 Fraud rate by transaction type

Which transaction types contain fraud? Helps determine whether transaction type is informative and explains why the original project focuses on TRANSFER and CASH_OUT.

In [8]:
sql = """
SELECT
    type,
    COUNT(*) AS total_txns,
    SUM(isFraud) AS fraud_txns,
    ROUND(100.0 * SUM(isFraud) / COUNT(*), 4) AS fraud_rate_pct
FROM transactions
GROUP BY type
ORDER BY fraud_rate_pct DESC
"""
result = con.execute(sql).df()
display(result)

,type,total_txns,fraud_txns,fraud_rate_pct
0,TRANSFER,532909,4097.0,0.7688
1,CASH_OUT,2237500,4116.0,0.1840
2,CASH_IN,1399284,0.0,0.0000
3,PAYMENT,2151495,0.0,0.0000
4,DEBIT,41432,0.0,0.0000


## 1.2 Built-in rule effectiveness

How many frauds were caught or missed by the original `isFlaggedFraud` rule? This is a rule-based reference point, not an equivalent held-out ML benchmark.

In [9]:
sql = """
SELECT
    SUM(isFraud) AS total_frauds,
    SUM(isFlaggedFraud) AS total_flagged,
    SUM(CASE WHEN isFraud = 1 AND isFlaggedFraud = 1 THEN 1 ELSE 0 END) AS frauds_caught_by_rule,
    SUM(CASE WHEN isFraud = 1 AND isFlaggedFraud = 0 THEN 1 ELSE 0 END) AS frauds_missed_by_rule
FROM transactions
"""
result = con.execute(sql).df()
display(result)

,total_frauds,total_flagged,frauds_caught_by_rule,frauds_missed_by_rule
0,8213.0,16.0,16.0,8197.0


## 1.3 Balance-draining signal

Among TRANSFER and CASH_OUT transactions, how often does the sender’s positive balance become zero? This motivates examining origin-account balance behavior.

In [10]:
sql = """
SELECT
    isFraud,
    COUNT(*) AS txns,
    SUM(CASE WHEN oldbalanceOrg > 0 AND newbalanceOrig = 0 THEN 1 ELSE 0 END) AS drained_count,
    ROUND(100.0 * SUM(CASE WHEN oldbalanceOrg > 0 AND newbalanceOrig = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS drained_pct
FROM transactions
WHERE type IN ('TRANSFER', 'CASH_ÅOUT')
GROUP BY isFraud
"""
result = con.execute(sql).df()
display(result)

,isFraud,txns,drained_count,drained_pct
0,0,2762196,1180062.0,42.72
1,1,8213,8012.0,97.55


## 1.4 Destination-balance anomaly

How often are both recorded destination balances zero despite a positive transfer amount? This is a potential anomaly in this simulated dataset.

In [11]:
sql = """
SELECT
    isFraud,
    COUNT(*) AS txns,
    SUM(CASE WHEN oldbalanceDest = 0 AND newbalanceDest = 0 AND amount > 0 THEN 1 ELSE 0 END) AS dest_zero_count,
    ROUND(100.0 * SUM(CASE WHEN oldbalanceDest = 0 AND newbalanceDest = 0 AND amount > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS dest_zero_pct
FROM transactions
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY isFraud
"""
result = con.execute(sql).df()
display(result)

,isFraud,txns,dest_zero_count,dest_zero_pct
0,1,8213,4070.0,49.56
1,0,2762196,1706.0,0.06


## 1.5 Mule chain: matching destination to cash-out origin

Does a fraudulent TRANSFER destination appear as the origin of any CASH_OUT? **Note:** the original SQL does not impose a later-time condition, so this does not prove a subsequent cash-out.

In [12]:
sql = """
SELECT
    COUNT(DISTINCT t.nameDest) AS fraud_transfer_destinations,
    COUNT(DISTINCT c.nameOrig) AS destinations_that_cashed_out
FROM transactions t
LEFT JOIN transactions c
       ON c.nameOrig = t.nameDest
      AND c.type = 'CASH_OUT'
WHERE t.type = 'TRANSFER'
  AND t.isFraud = 1
"""
result = con.execute(sql).df()
display(result)

,fraud_transfer_destinations,destinations_that_cashed_out
0,4097,3


## 1.6 Fraud concentration by transaction size

Divide TRANSFER and CASH_OUT amounts into ten roughly equal-sized groups and compare fraud counts and rates. This tests whether amount carries predictive information.

In [13]:
sql = """
WITH banded AS (
    SELECT
        amount,
        isFraud,
        NTILE(10) OVER (ORDER BY amount) AS amount_decile
    FROM transactions
    WHERE type IN ('TRANSFER', 'CASH_OUT')
)
SELECT
    amount_decile,
    COUNT(*) AS txns,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    SUM(isFraud) AS frauds,
    ROUND(100.0 * SUM(isFraud) / COUNT(*), 3) AS fraud_rate_pct
FROM banded
GROUP BY amount_decile
ORDER BY amount_decile
"""
result = con.execute(sql).df()
display(result)

,amount_decile,txns,min_amount,max_amount,frauds,fraud_rate_pct
0,1,277041,0.00,34162.47,759.0,0.274
1,2,277041,34162.55,66797.11,523.0,0.189
2,3,277041,66797.17,99390.69,415.0,0.150
3,4,277041,99390.91,133738.00,427.0,0.154
4,5,277041,133738.07,171260.93,379.0,0.137
5,6,277041,171261.63,215050.37,346.0,0.125
6,7,277041,215050.46,270842.40,403.0,0.145
7,8,277041,270843.23,352961.87,474.0,0.171
8,9,277041,352962.17,544867.83,763.0,0.275
9,10,277040,544869.79,92445516.64,3724.0,1.344
